# Regressão Logística - Previsão de Churn

**FGV - Modelagem Preditiva**

**Grupo:** João Victor Fortunato Teixeira Mundim, Martinho Aloisio Lenz, Rodrigo Pimentel, Bruno Barradas

Este notebook implementa Regressão Logística para prever Churn de clientes de Internet.

In [ ]:
####################################
# REGRESSÃO LOGÍSTICA - CHURN      #
####################################

# ===============================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ===============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings("ignore")

print("Bibliotecas importadas com sucesso.")

In [ ]:
# ============================================================
# 2. CONFIGURAÇÃO - Monte o Google Drive ou ajuste os caminhos
# ============================================================

# Para Google Colab, descomente as linhas abaixo:
# from google.colab import drive
# drive.mount('/content/drive')
# CAMINHO_BASE = '/content/drive/MyDrive/aula-predicao/atividade_avaliacao/dados/'

# Para execução local:
CAMINHO_BASE = '../dados/'

ARQUIVO_TREINO = CAMINHO_BASE + 'Churn_de_clientes_de_Internet_-_dados_de_treinamento_8122.xlsx'
ARQUIVO_CLASSIFICAR = CAMINHO_BASE + 'Clientes_reais_para_serem_classificados_c7fc.xlsx'

print(f"Caminho do arquivo de treino: {ARQUIVO_TREINO}")
print(f"Caminho do arquivo para classificar: {ARQUIVO_CLASSIFICAR}")

In [ ]:
# =============================================
# 3. CARREGAMENTO DOS DADOS DE TREINAMENTO
# =============================================
print("=== Carregando dados de treinamento ===")
df_treino = pd.read_excel(ARQUIVO_TREINO)

print(f"Dimensões dos dados: {df_treino.shape[0]} linhas x {df_treino.shape[1]} colunas")
print(f"\nColunas disponíveis:")
for i, col in enumerate(df_treino.columns):
    print(f"  {i+1}. {col} ({df_treino[col].dtype})")

display(df_treino.head())

In [ ]:
# =============================================
# 4. ANÁLISE EXPLORATÓRIA
# =============================================
print("=== Análise Exploratória ===")

print("\nDistribuição da Classe (Churn):")
print(df_treino['Classe'].value_counts())
print(f"\nProporção de Churn: {df_treino['Classe'].mean()*100:.1f}%")

print("\nValores Ausentes:")
missing = df_treino.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    for col in missing_cols.index:
        print(f"  {col}: {missing_cols[col]} valores ausentes ({missing_cols[col]/len(df_treino)*100:.1f}%)")
else:
    print("  Nenhum valor ausente encontrado.")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_treino['Classe'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição de Churn')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Quantidade')
axes[0].set_xticklabels(['Não Churn (0)', 'Churn (1)'], rotation=0)

df_treino['Classe'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[1].set_title('Proporção de Churn')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# 5. PRÉ-PROCESSAMENTO
# =====================================================================
print("=== Pré-processamento ===")

# Remover coluna ID
colunas_remover = ['ID_Cliente']
df_proc = df_treino.drop(columns=colunas_remover, errors='ignore').copy()

# Separar features e target
X = df_proc.drop(columns=['Classe']).copy()
y = df_proc['Classe'].copy()

colunas_features = list(X.columns)
print(f"\nFeatures utilizadas: {colunas_features}")

# Identificar colunas categóricas e numéricas
colunas_cat = X.select_dtypes(include=['object']).columns.tolist()
colunas_num = X.select_dtypes(include=['number']).columns.tolist()

print(f"\nColunas categóricas: {colunas_cat}")
print(f"Colunas numéricas: {colunas_num}")

# Tratamento de valores ausentes
medianas = {}
for col in colunas_num:
    if X[col].isnull().sum() > 0:
        mediana = X[col].median()
        medianas[col] = mediana
        X[col] = X[col].fillna(mediana)
        print(f"  {col}: valores ausentes preenchidos com mediana = {mediana:.2f}")

for col in colunas_cat:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna('Desconhecido')
        print(f"  {col}: valores ausentes preenchidos com 'Desconhecido'")

print("\nPré-processamento concluído.")

In [ ]:
# =====================================================================
# 6. ONE-HOT ENCODING PARA VARIÁVEIS CATEGÓRICAS
# =====================================================================
print("=== One-Hot Encoding ===")
print("\nA Regressão Logística requer One-Hot Encoding para variáveis categóricas.")

print("\nTabela ANTES da transformação:")
display(X.head())

# Aplicar One-Hot Encoding
X_encoded = pd.get_dummies(X, columns=colunas_cat, drop_first=True)

print("\nTabela APÓS One-Hot Encoding:")
display(X_encoded.head())

print(f"\nDimensões originais: {X.shape}")
print(f"Dimensões após encoding: {X_encoded.shape}")
print(f"\nNovas colunas criadas: {list(X_encoded.columns)}")

colunas_encoded = list(X_encoded.columns)

In [ ]:
# =====================================================================
# 7. NORMALIZAÇÃO DAS FEATURES
# =====================================================================
print("=== Normalização (StandardScaler) ===")
print("\nA Regressão Logística se beneficia de features normalizadas.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_scaled = pd.DataFrame(X_scaled, columns=X_encoded.columns)

print("\nEstatísticas após normalização:")
print(f"  Média (aproximada): {X_scaled.mean().mean():.4f}")
print(f"  Desvio padrão (aproximado): {X_scaled.std().mean():.4f}")

print("\nNormalização concluída.")

In [ ]:
# ============================================================
# 8. PARÂMETROS DO MODELO
# ============================================================
print("=== Parâmetros da Regressão Logística ===")

# Regularização
penalty = 'l2'

# Inverso da força de regularização
C = 1.0

# Solver
solver = 'lbfgs'

# Máximo de iterações
max_iter = 1000

# Proporção para validação
test_size = 0.2

# Semente aleatória
random_state = 42

print(f"\nParâmetros configurados:")
print(f"  penalty    = {penalty}")
print(f"  C          = {C}")
print(f"  solver     = {solver}")
print(f"  max_iter   = {max_iter}")
print(f"  test_size  = {test_size}")

In [ ]:
# ==========================================
# 9. DIVISÃO TREINO/TESTE E TREINAMENTO
# ==========================================
print("=== Treinando o modelo ===")

# Dividir dados
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=test_size, random_state=random_state, stratify=y
)

print(f"\nDados de treino: {len(X_train)} amostras")
print(f"Dados de teste: {len(X_test)} amostras")

# Criar e treinar modelo
modelo = LogisticRegression(
    penalty=penalty,
    C=C,
    solver=solver,
    max_iter=max_iter,
    random_state=random_state
)

modelo.fit(X_train, y_train)
print("\nModelo treinado com sucesso!")

In [ ]:
# ==========================================
# 10. AVALIAÇÃO DO MODELO
# ==========================================
print("=== Avaliação do Modelo ===")

# Predições
y_pred_train = modelo.predict(X_train)
y_pred_test = modelo.predict(X_test)
y_proba_test = modelo.predict_proba(X_test)[:, 1]

# Métricas
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)
auc_test = roc_auc_score(y_test, y_proba_test)

print(f"\nAcurácia no treino: {acc_train*100:.2f}%")
print(f"Acurácia no teste: {acc_test*100:.2f}%")
print(f"AUC-ROC no teste: {auc_test:.4f}")

# Validação Cruzada
cv_scores = cross_val_score(modelo, X_scaled, y, cv=5, scoring='accuracy')
print(f"\nValidação Cruzada (5-fold):")
print(f"  Média: {cv_scores.mean()*100:.2f}%")
print(f"  Desvio Padrão: {cv_scores.std()*100:.2f}%")

print("\nRelatório de Classificação (Teste):")
print(classification_report(y_test, y_pred_test, target_names=['Não Churn (0)', 'Churn (1)']))

In [ ]:
# ==========================================
# 11. VISUALIZAÇÕES
# ==========================================
print("=== Visualizações ===")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Não Churn', 'Churn'],
            yticklabels=['Não Churn', 'Churn'])
axes[0].set_title('Matriz de Confusão - Regressão Logística')
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
axes[1].plot(fpr, tpr, color='purple', lw=2, label=f'ROC (AUC = {auc_test:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[1].set_xlabel('Taxa de Falso Positivo')
axes[1].set_ylabel('Taxa de Verdadeiro Positivo')
axes[1].set_title('Curva ROC - Regressão Logística')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 12. ANÁLISE DOS COEFICIENTES
# ==========================================
print("=== Análise dos Coeficientes ===")

coeficientes = pd.DataFrame({
    'Feature': colunas_encoded,
    'Coeficiente': modelo.coef_[0]
})

coeficientes['Abs_Coef'] = np.abs(coeficientes['Coeficiente'])
coeficientes = coeficientes.sort_values('Abs_Coef', ascending=False).reset_index(drop=True)

print(f"\nIntercepto: {modelo.intercept_[0]:.4f}")
print("\nCoeficientes (ordenados por importância):")
display(coeficientes[['Feature', 'Coeficiente']].head(15))

# Gráfico dos coeficientes
fig, ax = plt.subplots(figsize=(10, 8))
top_coef = coeficientes.head(15)
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in top_coef['Coeficiente']]
ax.barh(range(len(top_coef)), top_coef['Coeficiente'], color=colors)
ax.set_yticks(range(len(top_coef)))
ax.set_yticklabels(top_coef['Feature'])
ax.invert_yaxis()
ax.axvline(x=0, color='black', linestyle='-', lw=0.5)
ax.set_xlabel('Coeficiente')
ax.set_title('Top 15 Coeficientes - Regressão Logística\n(Vermelho = aumenta chance de Churn, Verde = diminui)')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 13. INTERPRETAÇÃO DOS ODDS RATIOS
# ==========================================
print("=== Odds Ratios ===")
print("\nO Odds Ratio indica quanto a chance de Churn aumenta/diminui")
print("para cada aumento de 1 unidade (padronizada) na variável.")

odds_ratios = pd.DataFrame({
    'Feature': colunas_encoded,
    'Coeficiente': modelo.coef_[0],
    'Odds_Ratio': np.exp(modelo.coef_[0])
})

odds_ratios['Interpretacao'] = odds_ratios['Odds_Ratio'].apply(
    lambda x: f"+{(x-1)*100:.1f}% chance de Churn" if x > 1 else f"{(x-1)*100:.1f}% chance de Churn"
)

odds_ratios = odds_ratios.sort_values('Odds_Ratio', ascending=False).reset_index(drop=True)

print("\nVariáveis que AUMENTAM chance de Churn (Odds Ratio > 1):")
display(odds_ratios[odds_ratios['Odds_Ratio'] > 1][['Feature', 'Odds_Ratio', 'Interpretacao']].head(10))

print("\nVariáveis que DIMINUEM chance de Churn (Odds Ratio < 1):")
display(odds_ratios[odds_ratios['Odds_Ratio'] < 1][['Feature', 'Odds_Ratio', 'Interpretacao']].head(10))

In [ ]:
# ==========================================
# 14. TREINAR MODELO FINAL COM TODOS OS DADOS
# ==========================================
print("=== Treinando modelo final com todos os dados ===")

modelo_final = LogisticRegression(
    penalty=penalty,
    C=C,
    solver=solver,
    max_iter=max_iter,
    random_state=random_state
)

modelo_final.fit(X_scaled, y)
print("Modelo final treinado com 100% dos dados.")

In [ ]:
# ===================================================
# 15. CLASSIFICAR CLIENTES REAIS
# ===================================================
print("=== Carregando clientes para classificação ===")

try:
    df_classificar = pd.read_excel(ARQUIVO_CLASSIFICAR)
    print(f"Arquivo carregado: {ARQUIVO_CLASSIFICAR}")
    print(f"Total de clientes para classificar: {len(df_classificar)}")
    
    # Guardar IDs
    ids_clientes = df_classificar['ID_Cliente'].copy() if 'ID_Cliente' in df_classificar.columns else range(1, len(df_classificar)+1)
    
    # Preparar features
    X_novo = df_classificar.drop(columns=['ID_Cliente', 'Classe'], errors='ignore').copy()
    
    # Tratamento de valores ausentes
    for col in colunas_num:
        if col in X_novo.columns and col in medianas:
            X_novo[col] = X_novo[col].fillna(medianas[col])
    
    for col in colunas_cat:
        if col in X_novo.columns:
            X_novo[col] = X_novo[col].fillna('Desconhecido')
    
    # One-Hot Encoding
    X_novo_encoded = pd.get_dummies(X_novo, columns=colunas_cat, drop_first=True)
    
    # Garantir mesmas colunas que o treino
    for col in colunas_encoded:
        if col not in X_novo_encoded.columns:
            X_novo_encoded[col] = 0
    X_novo_encoded = X_novo_encoded[colunas_encoded]
    
    # Normalização
    X_novo_scaled = scaler.transform(X_novo_encoded)
    
    # Predições
    predicoes = modelo_final.predict(X_novo_scaled)
    probabilidades = modelo_final.predict_proba(X_novo_scaled)
    
    # Resultados
    resultados = pd.DataFrame({
        'ID_Cliente': ids_clientes,
        'Classe_Predita': predicoes,
        'Probabilidade_Nao_Churn': probabilidades[:, 0],
        'Probabilidade_Churn': probabilidades[:, 1]
    })
    
    resultados['Interpretacao'] = resultados['Classe_Predita'].apply(
        lambda x: 'CHURN - Cliente em risco' if x == 1 else 'NÃO CHURN - Cliente retido'
    )
    
    print("\n=== RESULTADOS DA CLASSIFICAÇÃO ===")
    display(resultados)
    
    print(f"\nResumo:")
    print(f"  Clientes classificados como Churn: {(predicoes == 1).sum()}")
    print(f"  Clientes classificados como Não Churn: {(predicoes == 0).sum()}")
    
except FileNotFoundError:
    print(f"Arquivo não encontrado: {ARQUIVO_CLASSIFICAR}")
    print("Pulando etapa de classificação de clientes reais.")

In [ ]:
# ==========================================
# 16. RESUMO FINAL
# ==========================================
print("="*60)
print("RESUMO - REGRESSÃO LOGÍSTICA")
print("="*60)
print(f"\nModelo: Regressão Logística")
print(f"Dataset: Churn de Clientes de Internet")
print(f"Total de registros de treinamento: {len(df_treino)}")
print(f"\nParâmetros do Modelo:")
print(f"  - Regularização: {penalty}")
print(f"  - C (força regularização): {C}")
print(f"  - Solver: {solver}")
print(f"\nDesempenho:")
print(f"  - Acurácia (treino): {acc_train*100:.2f}%")
print(f"  - Acurácia (teste): {acc_test*100:.2f}%")
print(f"  - AUC-ROC: {auc_test:.4f}")
print(f"  - Validação Cruzada: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")
print(f"\nTop 3 Variáveis Mais Influentes:")
for i, row in coeficientes.head(3).iterrows():
    sinal = "+" if row['Coeficiente'] > 0 else ""
    print(f"  {i+1}. {row['Feature']}: {sinal}{row['Coeficiente']:.4f}")
print("="*60)